<a href="https://colab.research.google.com/github/Apurva-J509/Classification_From_Scratch/blob/main/Linear_Discriminant_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [45]:
default = pd.read_excel('Finance_Default.xlsx', usecols = ['default', 'student', 'balance', 'income'])

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [46]:
X = default.iloc[:, 1:]
y = default.iloc[:, 0]

In [47]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X.iloc[:, 0] = le.fit_transform(X.iloc[:, 0])
y = le.fit_transform(y)

In [48]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [49]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.fit_transform(X_test)

In [50]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
model = LDA()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1924    2]
 [  58   16]]


0.97

###Linear Discriminant Analysis for p =1

In [51]:
import numpy as np


class LDAClassifier:

  def __init__(self):
    self.means = None
    self.prior_prb = None
    self.var = None

  def fit(self, X_train, y_train):
    # Ensure proper numpy array shapes
    X_train = np.array(X_train).reshape(-1, 1)
    y_train = np.array(y_train).reshape(-1, 1)

    n = X_train.shape[0]
    labels = np.unique(y_train)
    K = len(labels)

    # Split data into groups and compute means & priors
    self.means = np.zeros(K)
    self.prior_prb = np.zeros(K)
    labelled_groups = []

    for idx, i in enumerate(labels):
      group_idx = np.where(y_train.flatten() == i)[0]
      group = X_train[group_idx]
      labelled_groups.append(group)

      self.means[idx] = group.mean()
      self.prior_prb[idx] = group.shape[0] / n

    # Calculate pooled common variance (sigma^2)
    squared_sum = 0
    for idx, i in enumerate(labels):
      squared_sum += np.sum((labelled_groups[idx] - self.means[idx]) ** 2)

    self.var = (1 / (n - K)) * squared_sum
    return self

  def predict(self, X_test):
    X_test = np.array(X_test).reshape(-1, 1)

    # Vectorized discriminant score calculation for p = 1:
    # Formula per class k: delta_k(x) = (x * mean_k) / var - (mean_k^2) / (2 * var) + log(prior_k)
    # Shape of X_test: (N_test, 1)
    # Shape of self.means: (K,)

    # Term 1: (X_test @ means) / var -> shape: (N_test, K)
    term1 = (X_test @ self.means.reshape(1, -1)) / self.var

    # Term 2: - (means^2) / (2 * var) -> shape: (K,)
    term2 = -(self.means**2) / (2 * self.var)

    # Term 3: log(prior_prb) -> shape: (K,)
    term3 = np.log(self.prior_prb)

    # Combine all terms
    discriminants = term1 + term2 + term3

    # Return the class index with the maximum discriminant value for each sample
    return np.argmax(discriminants, axis=1)

In [52]:
lda = LDAClassifier()
lda.fit(X_train['income'], y_train)
y_pred = lda.predict(X_test['income'])

In [53]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1926    0]
 [  74    0]]


0.963

### Linear Discriminant Analysis for p>1

In [54]:
X_train = X_train.astype(float)

In [55]:
X_test = X_test.astype(float)

In [56]:
X_train.dtypes

,0
student,float64
balance,float64
income,float64


In [57]:
import numpy as np


class LDAClassifier:

  def __init__(self):
    self.means = None
    self.prior_prb = None
    self.cov_inv = None  # Store the inverted covariance matrix directly

  def fit(self, X_train, y_train):
    # Ensure proper numpy array shapes
    X_train = (
        np.array(X_train)
        if np.ndim(X_train) == 2
        else np.array(X_train).reshape(-1, 1)
    )
    y_train = np.array(y_train).reshape(-1, 1)

    n_samples, n_predictors = X_train.shape
    labels = np.unique(y_train)
    K = len(labels)

    self.means = np.zeros((K, n_predictors))
    self.prior_prb = np.zeros(K)

    # Compute class means and prior probabilities efficiently
    for idx, label in enumerate(labels):
      group = X_train[y_train.flatten() == label]
      self.means[idx] = group.mean(axis=0)
      self.prior_prb[idx] = group.shape[0] / n_samples

    # Compute pooled covariance matrix and its pseudo-inverse
    cov = np.cov(X_train, rowvar=False)
    # Handle single-feature edge case where np.cov returns a scalar
    if n_predictors == 1 and cov.ndim == 0:
      cov = np.array([[cov]])

    self.cov_inv = np.linalg.pinv(cov)
    return self

  def predict(self, X_test):
    X_test = (
        np.array(X_test)
        if np.ndim(X_test) == 2
        else np.array(X_test).reshape(-1, 1)
    )

    # Vectorized discriminant function calculation across all test samples:
    # Formula per class k: delta_k(x) = x^T @ inv(cov) @ mu_k - 0.5 * mu_k^T @ inv(cov) @ mu_k + log(prior_k)
    # Shape of X_test: (N_test, p)
    # Shape of self.means: (K, p)

    # Term 1: X_test @ inv(cov) @ mu_k^T  -> shape: (N_test, K)
    term1 = X_test @ self.cov_inv @ self.means.T

    # Term 2: -0.5 * diag(mu_k @ inv(cov) @ mu_k^T) -> shape: (K,)
    term2 = -0.5 * np.sum((self.means @ self.cov_inv) * self.means, axis=1)

    # Term 3: log(prior_prb) -> shape: (K,)
    term3 = np.log(self.prior_prb)

    # Combine all terms to get discriminants of shape (N_test, K)
    discriminants = term1 + term2 + term3

    # Return the class index with the maximum discriminant value for each sample
    return np.argmax(discriminants, axis=1)

In [58]:
lda = LDAClassifier()
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)

In [59]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1925    1]
 [  65    9]]


0.967